In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!pip install -q torch transformers peft bitsandbytes datasets accelerate

In [4]:
import json
import random

TEMPLATES = [
    ("billing_issue", "My card was charged twice for order {oid}, please refund immediately.", "high", True),
    ("billing_issue", "Why is there an extra service fee on my invoice for {oid}?", "medium", False),
    ("order_status", "Where is my package for order {oid}? Tracking hasn't moved.", "low", False),
    ("order_status", "Has order {oid} been dispatched yet?", "low", False),
    ("refund_request", "I want to return order {oid} and get a refund, item arrived damaged.", "medium", False),
    ("refund_request", "Cancel order {oid} and process full refund please.", "medium", False),
    ("account_access", "I cannot log into my account, password reset email is not arriving.", "high", True),
    ("account_access", "My account got locked after wrong OTP attempts.", "high", True),
    ("general_query", "Do you ship accessories outside India?", "low", False),
    ("general_query", "What are your customer support working hours?", "low", False),
]

def build_split(n: int, split_name: str, seed: int):
    rng = random.Random(seed)
    rows = []
    for i in range(n):
        intent, tmpl, urgency, human = rng.choice(TEMPLATES)
        has_order = rng.random() < 0.72
        oid = f"ORD-{2000 + i}" if has_order else None
        text = tmpl.format(oid=oid if oid else "my recent order")
        if split_name == "eval":
            text = f"{text} (Ref #{i+1})"
        rows.append({
            "ticket_text": text,
            "label": {
                "intent": intent,
                "urgency": urgency,
                "order_id": oid,
                "requires_human": human,
            },
        })
    return rows

train_rows = build_split(250, "train", seed=42)
eval_rows = build_split(50, "eval", seed=99)

with open("train.jsonl", "w", encoding="utf-8") as f:
    for r in train_rows:
        f.write(json.dumps(r) + "\n")

with open("eval_50.jsonl", "w", encoding="utf-8") as f:
    for r in eval_rows:
        f.write(json.dumps(r) + "\n")

print(f"Created train.jsonl ({len(train_rows)} rows) and eval_50.jsonl ({len(eval_rows)} rows)")

Created train.jsonl (250 rows) and eval_50.jsonl (50 rows)


In [5]:
import json
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LEN = 384

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def format_row(ex):
    prompt = (
        "<|im_start|>system\nOutput valid JSON only.<|im_end|>\n"
        f"<|im_start|>user\n{ex['ticket_text']}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    target = f"{json.dumps(ex['label'])}<|im_end|>"
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    t_ids = tokenizer(target, add_special_tokens=False)["input_ids"]
    return {
        "input_ids": (p_ids + t_ids)[:MAX_LEN],
        "labels": ([-100] * len(p_ids) + t_ids)[:MAX_LEN],
    }

def evaluate_50(records):
    model.eval()
    valid_json = 0
    intent_ok = 0
    exact_ok = 0
    errors = []

    for idx, r in enumerate(records):
        prompt = (
            "<|im_start|>system\nOutput valid JSON only.<|im_end|>\n"
            f"<|im_start|>user\n{r['ticket_text']}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        inp = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=100, do_sample=False)
        raw = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

        try:
            pred = json.loads(raw)
            valid_json += 1
            if pred.get("intent") == r["label"]["intent"]:
                intent_ok += 1
            if pred == r["label"]:
                exact_ok += 1
            elif len(errors) < 5:
                errors.append({"idx": idx, "gold": r["label"], "pred": pred})
        except Exception:
            if len(errors) < 5:
                errors.append({"idx": idx, "gold": r["label"], "raw": raw})

    n = len(records)
    metrics = {
        "n": n,
        "json_parse_rate": round(valid_json / n, 4),
        "intent_accuracy": round(intent_ok / n, 4),
        "exact_match": round(exact_ok / n, 4),
    }
    print(json.dumps(metrics, indent=2))
    return metrics, errors

with open("train.jsonl", "r", encoding="utf-8") as f:
    train_rows = [json.loads(line) for line in f]
with open("eval_50.jsonl", "r", encoding="utf-8") as f:
    eval_rows = [json.loads(line) for line in f]

print("=== BEFORE QLoRA ===")
before_metrics, _ = evaluate_50(eval_rows)

train_ds = Dataset.from_list(train_rows).map(format_row, remove_columns=["ticket_text", "label"])
args = TrainingArguments(
    output_dir="./qlora_ckpt",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
)
trainer.train()

print("=== AFTER QLoRA ===")
after_metrics, sample_errors = evaluate_50(eval_rows)
print("SAMPLE ERRORS:", json.dumps(sample_errors, indent=2))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (2 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
=== BEFORE QLoRA ===
{
  "n": 50,
  "json_parse_rate": 0.0,
  "intent_accuracy": 0.0,
  "exact_match": 0.0
}


Map:   0%|          | 0/250 [00:00<?, ? examples/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'warmup_ratio'

In [6]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

train_ds = Dataset.from_list(train_rows).map(format_row, remove_columns=["ticket_text", "label"])

args = TrainingArguments(
    output_dir="./qlora_ckpt",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=3,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8),
)
trainer.train()

print("=== AFTER QLoRA ===")
after_metrics, sample_errors = evaluate_50(eval_rows)
print("SAMPLE ERRORS:", json.dumps(sample_errors, indent=2))

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such a

Step,Training Loss
10,1.210113
20,0.109022
30,0.064459
40,0.056721


=== AFTER QLoRA ===
{
  "n": 50,
  "json_parse_rate": 1.0,
  "intent_accuracy": 1.0,
  "exact_match": 0.44
}
SAMPLE ERRORS: [
  {
    "idx": 0,
    "gold": {
      "intent": "account_access",
      "urgency": "high",
      "order_id": "ORD-2000",
      "requires_human": true
    },
    "pred": {
      "intent": "account_access",
      "order_id": null,
      "requires_human": true,
      "urgency": "high"
    }
  },
  {
    "idx": 1,
    "gold": {
      "intent": "general_query",
      "urgency": "low",
      "order_id": "ORD-2001",
      "requires_human": false
    },
    "pred": {
      "intent": "general_query",
      "order_id": null,
      "requires_human": false,
      "urgency": "low"
    }
  },
  {
    "idx": 4,
    "gold": {
      "intent": "account_access",
      "urgency": "high",
      "order_id": "ORD-2004",
      "requires_human": true
    },
    "pred": {
      "intent": "account_access",
      "order_id": "ORD-2041",
      "requires_human": true,
      "urgency": "high"